# Run ARC SFT Dataset Construction

Clones the ARC repository from GitLab, installs dependencies, creates `.env` from `.env.example`, and runs the SFT construction pipeline that executes the eight strategies on each scenario.

## 1. Runtime Parameters

Edit these values before running the notebook if needed.

In [ ]:
from pathlib import Path

REPO_URL = "https://gitlab.com/beryl.hoe/arc.git"
PROJECT_DIR = Path("/content/arc")
SCENARIOS_PATH = PROJECT_DIR / "data" / "mmlu_med_scenarios.jsonl"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "sft"
N_QUESTIONS = None  # None -> use the full scenario file
MODEL_NAME = None  # None -> use PKE_MODEL from .env
SEED = 42
TRAIN_RATIO = 0.8
VALIDATION_RATIO = 0.1
TEST_RATIO = 0.1
TEMPERATURE = None  # None -> use PKE_TEMPERATURE from .env
MAX_NEW_TOKENS = None  # None -> use PKE_MAX_NEW_TOKENS from .env
FORCE_RECLONE = False


## 2. Install System Packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-21-jdk-headless git git-lfs wget > /dev/null
!git lfs install


## 3. Clone the Repository

In [ ]:
import shutil
import subprocess

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project directory: {PROJECT_DIR}")


## 4. Install Python Dependencies

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)


## 5. Create and Load `.env`

The repository only tracks `.env.example`. This cell copies it to `.env` when needed.

In [ ]:
import os

env_path = PROJECT_DIR / ".env"
env_example_path = PROJECT_DIR / ".env.example"

if not env_path.exists():
    if not env_example_path.exists():
        raise FileNotFoundError(f"Missing both {env_path} and {env_example_path}")
    shutil.copyfile(env_example_path, env_path)
    print(f"Created {env_path} from {env_example_path}")
else:
    print(f"Using existing {env_path}")

def load_dotenv(path):
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")

load_dotenv(env_path)
for key in ["USE_DRIVE", "DRIVE_ROOT", "LOCAL_ROOT", "HF_TOKEN", "PKE_MODEL"]:
    print(f"{key}={os.environ.get(key, '')}")


## 6. Mount Google Drive

Drive must be mounted from the notebook kernel, not from a subprocess.

In [ ]:
use_drive = os.environ.get("USE_DRIVE", "false").lower() in {"1", "true", "yes", "y", "on"}
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("USE_DRIVE is false; skipping Google Drive mount.")


## 7. Optional Hugging Face Authentication

In [ ]:
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    subprocess.run(["huggingface-cli", "login", "--token", hf_token], check=True)
else:
    print("HF_TOKEN is empty; skipping Hugging Face login.")


## 8. Preflight Check

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))

preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        "from src.sft_dataset import STRATEGY_ORDER; print('python ok'); print(STRATEGY_ORDER)",
    ],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print("--- STDOUT ---")
print(preflight.stdout)
print("--- STDERR ---")
print(preflight.stderr)
if preflight.returncode != 0:
    raise RuntimeError(f"Preflight failed with exit code {preflight.returncode}")


## 9. Run the SFT Pipeline

In [ ]:
default_model = os.environ.get("PKE_MODEL", "Qwen/Qwen2.5-3B-Instruct")
default_temperature = os.environ.get("PKE_TEMPERATURE", "0.8")
default_max_new_tokens = os.environ.get("PKE_MAX_NEW_TOKENS", "256")

cmd = [sys.executable, "-u", "-m", "src.sft_dataset", str(SCENARIOS_PATH), str(OUTPUT_DIR)]
if N_QUESTIONS is not None:
    cmd.extend(["--n-questions", str(N_QUESTIONS)])
if MODEL_NAME is not None:
    cmd.extend(["--model", MODEL_NAME])
if SEED is not None:
    cmd.extend(["--seed", str(SEED)])
if TRAIN_RATIO is not None:
    cmd.extend(["--train-ratio", str(TRAIN_RATIO)])
if VALIDATION_RATIO is not None:
    cmd.extend(["--validation-ratio", str(VALIDATION_RATIO)])
if TEST_RATIO is not None:
    cmd.extend(["--test-ratio", str(TEST_RATIO)])
cmd.extend(["--temperature", str(TEMPERATURE if TEMPERATURE is not None else default_temperature)])
cmd.extend(["--max-new-tokens", str(MAX_NEW_TOKENS if MAX_NEW_TOKENS is not None else default_max_new_tokens)])

print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"SFT construction failed with exit code {returncode}")


## 10. Inspect Output

In [ ]:
root_dir = Path(os.environ["DRIVE_ROOT"] if use_drive else os.environ["LOCAL_ROOT"])
sft_root = root_dir / "outputs" / "sft"

print(f"SFT root: {sft_root}")
if sft_root.exists():
    for path in sorted(sft_root.glob("*.jsonl")):
        print(f"- {path.name}: {path.stat().st_size / 1024:.1f} KiB")
        with path.open("r", encoding="utf-8") as handle:
            first_line = handle.readline().strip()
        if first_line:
            print(first_line[:2000])
else:
    print("No SFT output directory was found.")
